# Lab 5: DNS — Wire Format, Resolution, and Iterative Resolvers
**CMSC395 — Computer Networks**  
Estimated time: 4–5 hours | Points: 100

---

Work through each cell in order. Parts 2 and 3 build directly on Part 1.

> **Do not use** `dnspython`, `socket.gethostbyname`, or `socket.getaddrinfo` for any resolution in Parts 1–3.  
> **Submission:** use **Kernel → Restart Kernel and Run All Cells** before your final push.

## Setup — Imports and Constants

In [ ]:
# Setup — run this cell first
import struct
import socket
import random
import time
import subprocess
import math
from pathlib import Path
from collections import defaultdict

import pyshark
import matplotlib.pyplot as plt
import numpy as np

# Root nameserver IPs (a, b, c root servers)
ROOT_SERVERS = [
    '198.41.0.4',    # a.root-servers.net
    '199.9.14.201',  # b.root-servers.net
    '192.33.4.12',   # c.root-servers.net
]

# Record type constants
QTYPE = {
    'A':     1,
    'NS':    2,
    'CNAME': 5,
    'PTR':   12,
    'MX':    15,
    'TXT':   16,
    'AAAA':  28,
}
QTYPE_NAME = {v: k for k, v in QTYPE.items()}

DNS_PCAP = Path('lab5_dns.pcap')

if not DNS_PCAP.exists():
    print(f'WARNING: {DNS_PCAP} not found — re-run nbgitpuller link')
else:
    print(f'DNS capture: {DNS_PCAP} ({DNS_PCAP.stat().st_size:,} bytes)')

print('Setup complete')

---
## Part 1: DNS Wire Format Parser

Build a parser that converts raw DNS message bytes into structured Python objects.
Complete the cells in order — each one builds on the previous.

### Cell 1.1 — Name Encoder

Implement `encode_name(name: str) -> bytes`.

Convert a domain name string to DNS wire format:
- Split on `.`
- Prefix each label with its 1-byte length
- Terminate with a zero byte
- The root domain (`.` or empty string) encodes as just `\x00`

Example: `encode_name('www.example.com')` → `b'\x03www\x07example\x03com\x00'`

In [ ]:
# Cell 1.1 — Name encoder

def encode_name(name: str) -> bytes:
    """
    Encode a domain name string to DNS wire format.
    encode_name('www.example.com') → b'\x03www\x07example\x03com\x00'
    encode_name('.')               → b'\x00'
    encode_name('')                → b'\x00'
    """
    # Your code here
    pass


# Tests
tests = [
    ('www.example.com', b'\x03www\x07example\x03com\x00'),
    ('com',             b'\x03com\x00'),
    ('.',               b'\x00'),
    ('',                b'\x00'),
    ('a.b.c.d',         b'\x01a\x01b\x01c\x01d\x00'),
]

all_pass = True
for name, expected in tests:
    got = encode_name(name)
    match = (got == expected)
    if not match:
        all_pass = False
    print(f'{"OK" if match else "FAIL"}: encode_name({name!r})')
    if not match:
        print(f'  Expected: {expected!r}')
        print(f'  Got:      {got!r}')

print('\nAll encoder tests passed!' if all_pass else '\nSome tests FAILED')


### Cell 1.2 — Name Decoder with Compression

Implement `decode_name(message: bytes, offset: int) -> tuple[str, int]`.

Read a DNS name from `message` starting at `offset`. Handle:
- Regular labels: read length byte, then that many characters
- Compression pointers: top two bits are `11`, followed offset into message
- Zero byte: end of name

Return `(name_string, new_offset)` where `new_offset` is the position **after** the
name in the original message (not after the pointer target).

**This is the hardest part.** Test it on real DNS responses before moving on.

In [ ]:
# Cell 1.2 — Name decoder with compression

def decode_name(message: bytes, offset: int) -> tuple:
    """
    Decode a DNS name from message starting at offset.
    Handles compression pointers.
    Returns (name: str, end_offset: int).
    end_offset is the position after the name/pointer in the original location.
    """
    labels = []
    visited = set()       # detect pointer loops
    jumped = False        # have we followed a pointer yet?
    end_offset = -1       # offset after the name in the original position

    while True:
        if offset >= len(message):
            raise ValueError(f'Name decode ran off end of message at offset {offset}')

        length = message[offset]

        if length == 0:
            # End of name
            if not jumped:
                end_offset = offset + 1
            break

        elif (length & 0xC0) == 0xC0:
            # Compression pointer
            if offset + 1 >= len(message):
                raise ValueError('Compression pointer extends beyond message')
            pointer = ((length & 0x3F) << 8) | message[offset + 1]
            if pointer in visited:
                raise ValueError(f'Compression pointer loop detected at {pointer}')
            visited.add(pointer)
            if not jumped:
                end_offset = offset + 2   # advance past the 2-byte pointer
            jumped = True
            offset = pointer

        elif (length & 0xC0) == 0:
            # Regular label
            offset += 1
            label = message[offset:offset + length].decode('ascii')
            labels.append(label)
            offset += length

        else:
            raise ValueError(f'Unknown label type: {length:#x} at offset {offset}')

    name = '.'.join(labels) if labels else '.'
    return name, end_offset


# Test 1: no compression
msg1 = encode_name('www.example.com')
name, off = decode_name(msg1, 0)
print(f'Test 1 (no compression): {name!r}, end_offset={off}')
assert name == 'www.example.com', f'Expected www.example.com, got {name}'
assert off == len(msg1), f'Expected offset {len(msg1)}, got {off}'

# Test 2: with compression pointer
# Manually construct a message with a pointer
# First name at offset 0: www.example.com
# Second name at offset 17: pointer back to offset 4 (example.com)
base_name = encode_name('www.example.com')          # 17 bytes
pointer_name = bytes([0xC0, 0x04])                  # pointer to offset 4
msg2 = base_name + pointer_name

name2, off2 = decode_name(msg2, 17)                 # read the pointer
print(f'Test 2 (with pointer): {name2!r}, end_offset={off2}')
assert name2 == 'example.com', f'Expected example.com, got {name2}'
assert off2 == 19, f'Expected offset 19, got {off2}'   # 17 + 2 bytes for pointer

# Test 3: real response from 8.8.8.8 (requires network)
print('\nTest 3: real response from 8.8.8.8...')
try:
    # Build minimal A query for 'dns.google'
    txid = random.randint(0, 65535)
    qname = encode_name('dns.google')
    header = struct.pack('!HHHHHH', txid, 0x0100, 1, 0, 0, 0)
    question = qname + struct.pack('!HH', 1, 1)
    query = header + question

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(3.0)
    sock.sendto(query, ('8.8.8.8', 53))
    response, _ = sock.recvfrom(4096)
    sock.close()

    # Try to decode the question section name
    name3, off3 = decode_name(response, 12)   # question starts at offset 12
    print(f'  Decoded question name: {name3!r}')
    assert 'google' in name3, 'Name should contain google'
    print('  Test 3 passed')
except Exception as e:
    print(f'  Test 3 skipped (network error): {e}')

print('\nDecoder implementation looks correct')


### Cell 1.3 — Header Parser

Implement `parse_header(message: bytes) -> dict`.

Parse the 12-byte DNS header. The flags field packs multiple sub-fields:
```
flags (16 bits): QR(1) | Opcode(4) | AA(1) | TC(1) | RD(1) | RA(1) | Z(1) | AD(1) | CD(1) | RCODE(4)
```
Extract each sub-field using bitwise operations.

In [ ]:
# Cell 1.3 — Header parser

def parse_header(message: bytes) -> dict:
    """
    Parse the 12-byte DNS message header.
    Returns dict with id, qr, opcode, aa, tc, rd, ra, rcode,
    qdcount, ancount, nscount, arcount.
    """
    if len(message) < 12:
        raise ValueError(f'Message too short for header: {len(message)} bytes')

    txid, flags, qdcount, ancount, nscount, arcount = struct.unpack('!HHHHHH', message[:12])

    # Extract flag sub-fields
    # Your code here
    qr     = (flags >> 15) & 0x1
    opcode = (flags >> 11) & 0xF
    aa     = (flags >> 10) & 0x1
    tc     = (flags >>  9) & 0x1
    rd     = (flags >>  8) & 0x1
    ra     = (flags >>  7) & 0x1
    rcode  = flags & 0xF

    return {
        'id':      txid,
        'qr':      qr,
        'opcode':  opcode,
        'aa':      aa,
        'tc':      tc,
        'rd':      rd,
        'ra':      ra,
        'rcode':   rcode,
        'qdcount': qdcount,
        'ancount': ancount,
        'nscount': nscount,
        'arcount': arcount,
    }


# Quick test with a known query
test_header = struct.pack('!HHHHHH',
    0x1234,   # ID
    0x0100,   # flags: QR=0, RD=1
    1, 0, 0, 0
)
h = parse_header(test_header + b'\x00' * 4)  # pad to at least 12 bytes
assert h['id'] == 0x1234
assert h['qr'] == 0
assert h['rd'] == 1
assert h['qdcount'] == 1
print('Header parser: OK')
print(f'Parsed: {h}')


### Cell 1.4 — Full Message Parser

Implement `parse_message(message: bytes) -> dict`.

Parse a complete DNS message by:
1. Parsing the header with `parse_header`
2. Parsing `qdcount` question records
3. Parsing `ancount` answer records
4. Parsing `nscount` authority records
5. Parsing `arcount` additional records

For resource records, decode RDATA based on the record type:
- A: 4 bytes → IPv4 string using `socket.inet_ntoa`
- AAAA: 16 bytes → IPv6 string using `socket.inet_ntop`
- NS, CNAME, PTR: compressed name
- MX: 2-byte preference + compressed name → `{'preference': n, 'exchange': name}`
- TXT: one or more length-prefixed strings concatenated
- Unknown: raw hex string

In [ ]:
# Cell 1.4 — Full message parser

def parse_rdata(message, offset, rdlength, rtype):
    """
    Parse RDATA for a given record type.
    Returns (data, new_offset) where data is a Python value.
    """
    end = offset + rdlength

    if rtype == 1:    # A
        data = socket.inet_ntoa(message[offset:offset + 4])

    elif rtype == 28:  # AAAA
        data = socket.inet_ntop(socket.AF_INET6, message[offset:offset + 16])

    elif rtype in (2, 5, 12):  # NS, CNAME, PTR
        data, _ = decode_name(message, offset)

    elif rtype == 15:  # MX
        preference = struct.unpack('!H', message[offset:offset + 2])[0]
        exchange, _ = decode_name(message, offset + 2)
        data = {'preference': preference, 'exchange': exchange}

    elif rtype == 16:  # TXT
        strings = []
        pos = offset
        while pos < end:
            str_len = message[pos]
            pos += 1
            strings.append(message[pos:pos + str_len].decode('utf-8', errors='replace'))
            pos += str_len
        data = ' '.join(strings)

    else:
        data = message[offset:end].hex()

    return data, end


def parse_record(message, offset):
    """
    Parse one resource record starting at offset.
    Returns (record_dict, new_offset).
    """
    name, offset = decode_name(message, offset)
    rtype, rclass, ttl, rdlength = struct.unpack('!HHIH', message[offset:offset + 10])
    offset += 10
    data, offset = parse_rdata(message, offset, rdlength, rtype)

    return {
        'name':  name,
        'type':  QTYPE_NAME.get(rtype, str(rtype)),
        'class': 'IN' if rclass == 1 else str(rclass),
        'ttl':   ttl,
        'data':  data,
    }, offset


def parse_question(message, offset):
    """
    Parse one question record starting at offset.
    Returns (question_dict, new_offset).
    """
    name, offset = decode_name(message, offset)
    qtype, qclass = struct.unpack('!HH', message[offset:offset + 4])
    return {
        'name':  name,
        'type':  QTYPE_NAME.get(qtype, str(qtype)),
        'class': 'IN' if qclass == 1 else str(qclass),
    }, offset + 4


def parse_message(message: bytes) -> dict:
    """
    Parse a complete DNS message.
    Returns a structured dict with header fields and all four sections.
    """
    result = parse_header(message)
    offset = 12

    # Parse questions
    questions = []
    for _ in range(result['qdcount']):
        q, offset = parse_question(message, offset)
        questions.append(q)
    result['questions'] = questions

    # Parse answer, authority, additional sections
    for section_key, count_key in [
        ('answers',    'ancount'),
        ('authority',  'nscount'),
        ('additional', 'arcount'),
    ]:
        records = []
        for _ in range(result[count_key]):
            rec, offset = parse_record(message, offset)
            records.append(rec)
        result[section_key] = records

    return result


print('parse_message defined. Verify in Cell 1.5 with real responses.')


### Cell 1.5 — Parser Verification

Send raw DNS queries to `8.8.8.8` and parse the responses with your parser.
Verify each result against `dig +short` output for the same name.

Test at least **3 different names** and **3 different record types**.

In [ ]:
# Cell 1.5 — Parser verification

def raw_query(name, qtype_str='A', server='8.8.8.8', port=53, timeout=3.0):
    """
    Send a raw DNS query and return the parsed response dict.
    Uses your encode_name and parse_message implementations.
    """
    qtype_num = QTYPE.get(qtype_str.upper(), 1)
    txid = random.randint(0, 65535)
    flags = 0x0100   # RD=1

    qname = encode_name(name)
    question = qname + struct.pack('!HH', qtype_num, 1)   # QCLASS=IN
    header = struct.pack('!HHHHHH', txid, flags, 1, 0, 0, 0)
    query = header + question

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(timeout)
    try:
        sock.sendto(query, (server, port))
        response, _ = sock.recvfrom(4096)
    finally:
        sock.close()

    parsed = parse_message(response)
    if parsed['id'] != txid:
        raise ValueError(f'Transaction ID mismatch: {parsed["id"]} != {txid}')
    return parsed


def dig(name, qtype='A'):
    """Run dig +short and return the output lines."""
    result = subprocess.run(
        ['dig', '+short', f'-t{qtype}', name],
        capture_output=True, text=True
    )
    return [l.strip() for l in result.stdout.strip().split('\n') if l.strip()]


# Test cases — at least 3 names, at least 3 record types
test_cases = [
    ('dns.google',    'A'),
    ('dns.google',    'AAAA'),
    ('gmail.com',     'MX'),
    ('google.com',    'NS'),
    ('google.com',    'TXT'),
]

for name, qtype in test_cases:
    print(f'\n=== {qtype} {name} ===')
    try:
        parsed = raw_query(name, qtype)
        dig_output = dig(name, qtype)

        print(f'Our parser ({len(parsed["answers"])} answers):')
        for rec in parsed['answers']:
            print(f'  {rec["type"]} {rec["data"]}  TTL={rec["ttl"]}')

        print(f'dig +short:')
        for line in dig_output:
            print(f'  {line}')

    except Exception as e:
        print(f'Error: {e}')


### Reflection 1.A

DNS name compression uses backward pointers only — a pointer always refers to an earlier
position in the message. Why is this guaranteed by the protocol design?
What would happen if forward pointers were allowed?

**Your answer:**

*(Write here)*

### Reflection 1.B

TXT records can contain multiple strings within a single record. Look at the TXT records
for a real domain using your resolver. What are they used for, and why might one record
contain multiple strings rather than using multiple separate records?

**Your answer:**

*(Write here — include example TXT record content from your query)*

---
## Part 2: Stub Resolver

Build a `resolve()` function that queries a recursive resolver and handles CNAME chains.
Do not use `socket.gethostbyname` or any library resolver.

### Cell 2.1 — Query Builder

Implement `build_query(name, record_type='A', recursion_desired=True) -> bytes`.

Build a complete DNS query message ready to send via UDP. Use a random 16-bit transaction ID.

In [ ]:
# Cell 2.1 — Query builder

def build_query(name: str, record_type: str = 'A', recursion_desired: bool = True) -> bytes:
    """
    Build a DNS query message for the given name and record type.
    Returns raw bytes ready to send via UDP.
    """
    txid  = random.randint(0, 65535)
    flags = 0x0100 if recursion_desired else 0x0000
    qtype = QTYPE.get(record_type.upper())
    if qtype is None:
        raise ValueError(f'Unknown record type: {record_type}')

    # Your code here
    pass


# Verify the query is parseable
q = build_query('www.example.com', 'A')
print(f'Query length: {len(q)} bytes')
header = parse_header(q)
print(f'Header: {header}')
assert header['qdcount'] == 1
assert header['rd'] == 1
question, _ = parse_question(q, 12)
print(f'Question: {question}')
assert question['name'] == 'www.example.com'
assert question['type'] == 'A'
print('Query builder: OK')


### Cell 2.2 — Stub Resolver

Implement `resolve(name, record_type, server, port, timeout)` with CNAME chain following.

- For PTR queries, accept an IP address string and convert to `x.x.x.x.in-addr.arpa`
- Follow CNAME chains up to `max_depth=10`
- Raise `DNSError` on NXDOMAIN, timeout, or unexpected response

In [ ]:
# Cell 2.2 — Stub resolver

class DNSError(Exception):
    """Raised on DNS resolution failure."""
    def __init__(self, message, rcode=None):
        self.rcode = rcode
        super().__init__(message)


RCODE_NAMES = {0: 'NOERROR', 1: 'FORMERR', 2: 'SERVFAIL',
               3: 'NXDOMAIN', 4: 'NOTIMP', 5: 'REFUSED'}


def ip_to_ptr_name(ip: str) -> str:
    """Convert '8.8.8.8' to '8.8.8.8.in-addr.arpa'."""
    return '.'.join(reversed(ip.split('.'))) + '.in-addr.arpa'


def resolve(name: str, record_type: str = 'A',
            server: str = '8.8.8.8', port: int = 53,
            timeout: float = 3.0, max_depth: int = 10) -> list:
    """
    Resolve name using a stub query to server.
    Returns list of answer record dicts.
    Follows CNAME chains automatically.
    Raises DNSError on failure.
    """
    # Handle PTR records
    if record_type.upper() == 'PTR':
        try:
            socket.inet_aton(name)   # is it an IP?
            name = ip_to_ptr_name(name)
        except socket.error:
            pass   # already a ptr name

    current_name = name
    for depth in range(max_depth):
        query = build_query(current_name, record_type)

        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        sock.settimeout(timeout)
        try:
            sock.sendto(query, (server, port))
            response, _ = sock.recvfrom(4096)
        except socket.timeout:
            raise DNSError(f'Timeout querying {server} for {current_name}')
        finally:
            sock.close()

        parsed = parse_message(response)

        if parsed['rcode'] == 3:
            raise DNSError(f'NXDOMAIN: {current_name} does not exist', rcode=3)
        if parsed['rcode'] != 0:
            rname = RCODE_NAMES.get(parsed['rcode'], str(parsed['rcode']))
            raise DNSError(f'{rname} from {server} for {current_name}',
                           rcode=parsed['rcode'])

        answers = parsed['answers']

        # Check for CNAME in answers when we want a different type
        if record_type.upper() != 'CNAME':
            cnames = [r for r in answers if r['type'] == 'CNAME']
            direct = [r for r in answers if r['type'] == record_type.upper()]
            if direct:
                return direct
            if cnames:
                current_name = cnames[0]['data']
                continue

        if answers:
            return answers

        raise DNSError(f'No answers for {current_name} {record_type}')

    raise DNSError(f'CNAME chain too long for {name}')


# Quick test
result = resolve('dns.google', 'A')
print(f'dns.google A: {[r["data"] for r in result]}')


### Cell 2.3 — Record Type Tests

Test all 7 record types. For each, print the answers in a human-readable format.

In [ ]:
# Cell 2.3 — Record type tests

queries = [
    ('www.google.com',  'A',     'IPv4 address'),
    ('www.google.com',  'AAAA',  'IPv6 address'),
    ('gmail.com',       'MX',    'Mail exchangers'),
    ('google.com',      'NS',    'Nameservers'),
    ('google.com',      'TXT',   'TXT records (SPF etc.)'),
    ('8.8.8.8',         'PTR',   'Reverse DNS'),
    ('www.github.com',  'CNAME', 'CNAME record'),
]

for name, qtype, description in queries:
    print(f'\n=== {qtype} {name} ({description}) ===')
    try:
        records = resolve(name, qtype)
        for r in records:
            if isinstance(r['data'], dict):
                print(f'  {r["type"]} pref={r["data"]["preference"]} {r["data"]["exchange"]}  TTL={r["ttl"]}')
            else:
                data_str = str(r['data'])[:80]
                print(f'  {r["type"]} {data_str}  TTL={r["ttl"]}')
    except DNSError as e:
        print(f'  DNSError: {e}')


### Cell 2.4 — CNAME Chain Demonstration

Find a name that resolves through at least 2 CNAME hops.
Show each step in the chain with TTL values.

Modify `resolve()` to return the full CNAME chain in addition to the final answer,
or implement a separate `resolve_chain()` function.

In [ ]:
# Cell 2.4 — CNAME chain demonstration

def resolve_chain(name: str, record_type: str = 'A',
                  server: str = '8.8.8.8') -> list:
    """
    Resolve name, returning the full chain of steps.
    Each step is a list of answer records at that step.
    """
    chain = []
    current_name = name
    max_depth = 10

    for depth in range(max_depth):
        # Your code here — query, collect answers, follow CNAMEs, record each step
        pass

    return chain


# Find a name with multiple CNAME hops
# Suggestions: docs.github.com, www.amazon.com, cdn names
chain_targets = ['docs.github.com', 'www.amazon.com']

for target in chain_targets:
    print(f'\nResolving chain for {target}:')
    try:
        chain = resolve_chain(target, 'A')
        for i, step in enumerate(chain, 1):
            for rec in step:
                print(f'  Step {i}: {rec["name"]} {rec["type"]} → {rec["data"]}  TTL={rec["ttl"]}')
        if len(chain) >= 2:
            print(f'  Chain length: {len(chain)} hops ✓')
    except DNSError as e:
        print(f'  Error: {e}')


### Reflection 2.A

Send a query with `RD=0` (recursion not desired) to `8.8.8.8` for a name you successfully
resolved with `RD=1`. What response do you get, and why is it different?

In [ ]:
# Test RD=0 query
q_rd0 = build_query('www.google.com', 'A', recursion_desired=False)

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(3.0)
sock.sendto(q_rd0, ('8.8.8.8', 53))
response, _ = sock.recvfrom(4096)
sock.close()

parsed_rd0 = parse_message(response)
print('RD=0 response:')
print(f'  rcode:   {parsed_rd0["rcode"]} ({RCODE_NAMES.get(parsed_rd0["rcode"])})')
print(f'  answers: {len(parsed_rd0["answers"])}')
print(f'  ra:      {parsed_rd0["ra"]}  (RA=1 means server supports recursion)')
print(f'  authority records: {len(parsed_rd0["authority"])}')


**Your answer:**

*(What did you observe? Why is RD=0 different from RD=1?)*

### Reflection 2.B

MX records include a preference value. Look at the MX records for a large mail domain.
What do multiple MX records with different preference values tell you about mail delivery?
What happens when the lowest-preference server is unreachable?

**Your answer:**

*(Write here — include MX records from your query)*

---
## Part 3: Iterative Resolver

Walk the DNS hierarchy yourself — from root servers to authoritative nameservers —
without asking a recursive resolver to do the work.

### Cell 3.1 — Iterative Resolver

Implement `iterative_resolve(name, record_type='A', max_hops=20)`.

At each step:
1. Query the current nameserver with `RD=0`
2. If the answer section is non-empty, return those records
3. If the authority section has NS records, extract nameserver names
4. Check the additional section for glue records (A records for those NS names)
5. If glue is available, use it directly; otherwise call `resolve()` to find the NS IP
6. Query the next nameserver and repeat

Log every step — the log is part of your grade.

In [ ]:
# Cell 3.1 — Iterative resolver

def query_nameserver(name: str, record_type: str,
                     server_ip: str, timeout: float = 3.0) -> dict:
    """
    Send a non-recursive query to server_ip.
    Returns parsed response dict.
    """
    query = build_query(name, record_type, recursion_desired=False)
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(timeout)
    try:
        sock.sendto(query, (server_ip, 53))
        response, _ = sock.recvfrom(4096)
    except socket.timeout:
        raise DNSError(f'Timeout querying {server_ip}')
    finally:
        sock.close()
    return parse_message(response)


def iterative_resolve(name: str, record_type: str = 'A',
                      max_hops: int = 20) -> dict:
    """
    Iterative DNS resolution starting from a root server.
    Returns {'answers': [...], 'log': [...], 'hops': n, 'time_sec': t}.
    """
    log = []
    start_time = time.perf_counter()

    # Start with a random root server
    current_servers = [random.choice(ROOT_SERVERS)]

    for hop in range(1, max_hops + 1):
        if not current_servers:
            raise DNSError('No nameservers to query')

        server_ip = current_servers[0]
        log.append(f'[{hop}] Querying {server_ip} for {name} ({record_type})')

        try:
            response = query_nameserver(name, record_type, server_ip)
        except DNSError as e:
            log.append(f'    → Error: {e} — trying next server')
            current_servers = current_servers[1:]
            continue

        if response['rcode'] == 3:
            log.append(f'    → NXDOMAIN')
            raise DNSError(f'NXDOMAIN: {name}', rcode=3)

        # Check for answers
        answers = response['answers']
        if answers:
            log.append(f'    → Answer: {[(r["type"], r["data"]) for r in answers]}')
            elapsed = time.perf_counter() - start_time
            return {
                'answers': answers,
                'log': log,
                'hops': hop,
                'time_sec': elapsed
            }

        # Look for referral in authority section
        ns_records = [r for r in response['authority'] if r['type'] == 'NS']
        if not ns_records:
            raise DNSError(f'No answer and no referral from {server_ip}')

        # Build glue map from additional section
        glue = {r['name'].rstrip('.'): r['data']
                for r in response['additional'] if r['type'] == 'A'}

        ns_names = [r['data'].rstrip('.') for r in ns_records]
        log.append(f'    → Referral: NS records for zone ({len(ns_names)} servers)')

        # Resolve next nameservers using glue or fallback to stub resolver
        next_servers = []
        for ns_name in ns_names[:4]:   # limit to 4 candidates
            if ns_name in glue:
                log.append(f'      {ns_name} → {glue[ns_name]} (glue)')
                next_servers.append(glue[ns_name])
            else:
                # Your code here — fall back to stub resolver for NS IP
                pass

        if not next_servers:
            raise DNSError(f'Could not resolve any nameserver for {name}')

        current_servers = next_servers

    raise DNSError(f'Max hops ({max_hops}) exceeded for {name}')


# Quick test
print('Testing iterative resolver on dns.google...')
result = iterative_resolve('dns.google', 'A')
print(f'\nResolution complete: {result["hops"]} hops, {result["time_sec"]:.3f}s')
print('\nResolution log:')
for line in result['log']:
    print(line)
print('\nAnswers:')
for r in result['answers']:
    print(f'  {r["type"]} {r["data"]}  TTL={r["ttl"]}')


### Cell 3.2 — Resolution Traces

Run iterative resolution for at least 5 domains. Display the full log for each.
Choose domains with interesting traces — deeply delegated names, CNAME chains,
or names requiring glue lookups.

In [ ]:
# Cell 3.2 — Resolution traces

trace_targets = [
    ('www.ietf.org',     'A'),
    ('mail.python.org',  'A'),
    ('ns1.google.com',   'A'),
    ('8.8.4.4',          'PTR'),
    ('gmail.com',        'MX'),
]

trace_results = {}

for name, qtype in trace_targets:
    print(f'\n{"="*60}')
    print(f'Resolving {name} ({qtype})')
    print('='*60)
    try:
        result = iterative_resolve(name, qtype)
        trace_results[(name, qtype)] = result
        for line in result['log']:
            print(line)
        print(f'\nResolution: {result["hops"]} hops, {result["time_sec"]:.3f}s')
        for r in result['answers']:
            data = r['data']
            if isinstance(data, dict):
                data = f"pref={data['preference']} {data['exchange']}"
            print(f'  {r["type"]} {data}  TTL={r["ttl"]}')
    except DNSError as e:
        print(f'Error: {e}')


### Cell 3.3 — Performance Comparison

Resolve the same 10 names using both `iterative_resolve` and `resolve` (querying 8.8.8.8).
Measure total resolution time for each. Plot a bar chart comparing the two.

In [ ]:
# Cell 3.3 — Performance comparison

perf_targets = [
    'www.google.com',
    'www.github.com',
    'www.python.org',
    'www.cloudflare.com',
    'www.mozilla.org',
    'www.wikipedia.org',
    'www.ietf.org',
    'dns.google',
    'one.one.one.one',
    'www.amazon.com',
]

iterative_times = []
stub_times = []
labels = []

for name in perf_targets:
    print(f'{name}...')
    labels.append(name.replace('www.', ''))

    # Iterative
    try:
        t0 = time.perf_counter()
        iterative_resolve(name, 'A')
        iterative_times.append(time.perf_counter() - t0)
    except Exception as e:
        print(f'  Iterative error: {e}')
        iterative_times.append(None)

    # Stub
    try:
        t0 = time.perf_counter()
        resolve(name, 'A')
        stub_times.append(time.perf_counter() - t0)
    except Exception as e:
        print(f'  Stub error: {e}')
        stub_times.append(None)

# Plot
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))

iter_vals = [t if t is not None else 0 for t in iterative_times]
stub_vals = [t if t is not None else 0 for t in stub_times]

ax.bar(x - width/2, iter_vals, width, label='Iterative (from root)')
ax.bar(x + width/2, stub_vals, width, label='Stub (via 8.8.8.8)')

ax.set_xlabel('Domain')
ax.set_ylabel('Resolution time (seconds)')
ax.set_title('Iterative vs Stub Resolver Performance')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nMean iterative: {np.mean([t for t in iterative_times if t]):.3f}s')
print(f'Mean stub:      {np.mean([t for t in stub_times if t]):.3f}s')


### Cell 3.4 — ASN Annotation

Implement `ip_to_asn(ip)` using Team Cymru's DNS service.
Query `{reversed_ip}.origin.asn.cymru.com` for TXT records using your stub resolver.
Annotate at least one resolution trace with AS numbers.

In [ ]:
# Cell 3.4 — ASN annotation

def ip_to_asn(ip: str) -> dict:
    """
    Look up ASN for an IP using Team Cymru's DNS-based service.
    Query: {reversed_ip}.origin.asn.cymru.com TXT
    Response TXT format: 'ASN | prefix | CC | registry | date'
    Returns dict with 'asn', 'prefix', 'country' or None on failure.
    """
    try:
        reversed_ip = '.'.join(reversed(ip.split('.')))
        query_name = f'{reversed_ip}.origin.asn.cymru.com'
        records = resolve(query_name, 'TXT')
        if not records:
            return None
        # TXT format: 'ASN | IP prefix | CC | registry | date'
        txt = records[0]['data']
        parts = [p.strip() for p in txt.split('|')]
        if len(parts) >= 3:
            return {
                'asn':     parts[0].strip(),
                'prefix':  parts[1].strip(),
                'country': parts[2].strip(),
            }
    except Exception:
        pass
    return None


# Test ASN lookup
test_ips = ['8.8.8.8', '1.1.1.1', '198.41.0.4']
print('ASN lookups:')
for ip in test_ips:
    asn = ip_to_asn(ip)
    print(f'  {ip}: {asn}')

# Annotate a resolution trace
print('\n=== Annotated trace for www.ietf.org ===')
try:
    result = iterative_resolve('www.ietf.org', 'A')
    for line in result['log']:
        # Extract IPs from log lines and annotate with ASN
        import re
        ips = re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', line)
        print(line)
        for ip in ips:
            asn_info = ip_to_asn(ip)
            if asn_info:
                print(f'      AS{asn_info["asn"]} {asn_info["prefix"]} ({asn_info["country"]})')
except DNSError as e:
    print(f'Error: {e}')


### Reflection 3.A

Compare hop counts across your resolution traces. Which domains required the most hops?
What does a high hop count suggest about that domain's DNS delegation structure?

**Your answer:**

*(Write here — reference specific domains and hop counts)*

### Reflection 3.B

Some referrals include glue records; others do not. What determines whether glue is included?
What problem would occur if glue records were never provided for a nameserver whose name
is within the delegated zone?

**Your answer:**

*(Write here)*

### Reflection 3.C

Your performance comparison shows iterative resolution is slower than querying 8.8.8.8.
What optimizations does a production recursive resolver use that your implementation lacks?

**Your answer:**

*(Write here)*

---
## Part 4: DNS Traffic Analysis

Analyze `lab5_dns.pcap` — a capture of real DNS traffic — to answer questions about
query distribution, caching behavior, response timing, and anomalies.

### Cell 4.1 — Query Distribution

How many DNS queries are in the capture? Show the distribution by record type
and the top 10 most-queried names.

In [ ]:
# Cell 4.1 — Query distribution

from collections import Counter

query_types   = Counter()
query_names   = Counter()
nxdomain_count = 0
total_queries  = 0
total_responses = 0

cap = pyshark.FileCapture(str(DNS_PCAP), display_filter='dns')
for pkt in cap:
    try:
        is_response = int(pkt.dns.flags_response)
        if not is_response:
            total_queries += 1
            qtype_num = int(pkt.dns.qry_type)
            qtype_str = QTYPE_NAME.get(qtype_num, str(qtype_num))
            query_types[qtype_str] += 1
            query_names[pkt.dns.qry_name] += 1
        else:
            total_responses += 1
            if int(pkt.dns.flags_rcode) == 3:
                nxdomain_count += 1
    except AttributeError:
        pass
cap.close()

print(f'Total queries:   {total_queries}')
print(f'Total responses: {total_responses}')
print(f'NXDOMAIN:        {nxdomain_count} ({100*nxdomain_count/max(total_responses,1):.1f}% of responses)')

print('\nQuery type distribution:')
for qtype, count in query_types.most_common():
    print(f'  {qtype:<8} {count:>6}')

print('\nTop 10 queried names:')
for name, count in query_names.most_common(10):
    print(f'  {count:>5}  {name}')


### Cell 4.2 — Caching Behavior

Find pairs of queries for the same name where the second query arrived before the
first response's TTL would have expired. Also find the longest and shortest TTLs.

In [ ]:
# Cell 4.2 — Caching behavior

# Build query/response pairs keyed by (transaction_id)
queries_by_id  = {}   # txid -> (name, time)
responses_by_id = {}  # txid -> (name, time, ttl)

cap = pyshark.FileCapture(str(DNS_PCAP), display_filter='dns')
for pkt in cap:
    try:
        t = float(pkt.frame_info.time_relative)
        txid = pkt.dns.id
        is_response = int(pkt.dns.flags_response)

        if not is_response:
            queries_by_id[txid] = {
                'name': pkt.dns.qry_name,
                'time': t,
                'type': pkt.dns.qry_type,
            }
        else:
            try:
                ttl = int(pkt.dns.resp_ttl)
            except AttributeError:
                ttl = None
            responses_by_id[txid] = {
                'name': pkt.dns.qry_name,
                'time': t,
                'ttl':  ttl,
            }
    except AttributeError:
        pass
cap.close()

# Your caching analysis here
# Find query pairs where second query < first response time + TTL


### Cell 4.3 — Response Timing

Compute response time for every query/response pair. Plot the distribution.
Identify the slowest query and examine it.

In [ ]:
# Cell 4.3 — Response timing

response_times = []   # list of (name, delay_ms)

for txid, q in queries_by_id.items():
    if txid in responses_by_id:
        r = responses_by_id[txid]
        delay_ms = (r['time'] - q['time']) * 1000
        if delay_ms > 0:
            response_times.append((q['name'], delay_ms))

delays = [d for _, d in response_times]
print(f'Transactions with matched response: {len(delays)}')
print(f'Mean response time:   {np.mean(delays):.2f} ms')
print(f'Median response time: {np.median(delays):.2f} ms')
print(f'99th percentile:      {np.percentile(delays, 99):.2f} ms')
print(f'Max response time:    {max(delays):.2f} ms')

# Slowest query
slowest = max(response_times, key=lambda x: x[1])
print(f'\nSlowest query: {slowest[0]} ({slowest[1]:.2f} ms)')

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(delays, bins=50, edgecolor='white')
ax.set_xlabel('Response time (ms)')
ax.set_ylabel('Count')
ax.set_title('DNS Response Time Distribution')
ax.axvline(np.median(delays), color='red', linestyle='--', label=f'Median {np.median(delays):.1f}ms')
ax.legend()
plt.tight_layout()
plt.show()


### Cell 4.4 — Anomaly Detection

Find queries with unusually long subdomain labels (> 40 chars) and
NXDOMAIN responses for high-entropy names. For the long-subdomain queries,
try to decode the subdomain as BASE64.

In [ ]:
# Cell 4.4 — Anomaly detection

import math

def shannon_entropy(s: str) -> float:
    """Compute Shannon entropy of a string in bits per character."""
    if not s:
        return 0.0
    freq = Counter(s.lower())
    total = len(s)
    return -sum((c/total) * math.log2(c/total) for c in freq.values())


long_subdomain_queries = []
high_entropy_nxdomains = []

for txid, q in queries_by_id.items():
    name = q['name']
    labels = name.split('.')

    # Find unusually long subdomain labels
    for label in labels[:-2]:   # skip TLD and SLD
        if len(label) > 40:
            long_subdomain_queries.append({
                'name':  name,
                'label': label,
                'len':   len(label),
                'entropy': shannon_entropy(label),
                'time':  q['time'],
            })
            break

    # Find high-entropy NXDOMAIN names
    if txid in responses_by_id:
        resp = responses_by_id[txid]
        # Check if this was NXDOMAIN — need rcode from the response
        # (stored separately below)

print(f'Queries with long subdomains (>40 chars): {len(long_subdomain_queries)}')
if long_subdomain_queries:
    print('\nLong subdomain queries:')
    for q in sorted(long_subdomain_queries, key=lambda x: x['len'], reverse=True)[:10]:
        print(f'  len={q["len"]} entropy={q["entropy"]:.2f} label={q["label"][:50]}...')
        # Try decoding as BASE64
        try:
            import base64 as _b64
            # Pad to multiple of 4
            padded = q['label'] + '=' * (4 - len(q['label']) % 4)
            decoded = _b64.b64decode(padded, validate=False)
            print(f'  Decoded: {decoded[:50]}')
        except Exception:
            print(f'  (Not valid BASE64)')


### Reflection 4.A

Your anomaly detection found queries with long, high-entropy subdomain labels.
Describe how DNS can be used as a covert data exfiltration channel.
What are the practical limitations of this technique, and what detection
strategies would a network operator use?

**Your answer:**

*(Write here)*

### Reflection 4.B

You found queries where the application bypassed its local DNS cache and re-queried
before the TTL expired. What are the legitimate reasons an application might do this?
When is this behavior a problem?

**Your answer:**

*(Write here)*

---
## Submission Checklist

Before your final push, complete this checklist. Replace each `[ ]` with `[x]` when done.

- [ ] Cell 1.1: Name encoder passes all tests
- [ ] Cell 1.2: Name decoder handles compression — all 3 tests pass
- [ ] Cell 1.3: Header parser produces correct fields
- [ ] Cell 1.4: Full message parser handles all record types
- [ ] Cell 1.5: Parser verified against dig for 3+ names and 3+ types
- [ ] Reflections 1.A and 1.B completed
- [ ] Cell 2.1: Query builder verified against parse_message
- [ ] Cell 2.2: Stub resolver works for all 7 record types
- [ ] Cell 2.3: All 7 record type tests shown with output
- [ ] Cell 2.4: CNAME chain with 2+ hops demonstrated
- [ ] Reflections 2.A and 2.B completed
- [ ] Cell 3.1: Iterative resolver produces resolution log
- [ ] Cell 3.2: Full traces shown for 5+ domains
- [ ] Cell 3.3: Performance comparison bar chart shown
- [ ] Cell 3.4: ASN lookup works, at least one trace annotated
- [ ] Reflections 3.A, 3.B, and 3.C completed
- [ ] Cell 4.1: Query distribution and top-10 table shown
- [ ] Cell 4.2: Caching analysis results shown
- [ ] Cell 4.3: Response time plot included, slowest query identified
- [ ] Cell 4.4: Long subdomain queries found and decoded
- [ ] Reflections 4.A and 4.B completed
- [ ] Notebook runs cleanly with Kernel → Restart Kernel and Run All Cells
- [ ] At least 5 commits with descriptive messages

Final commit message must be exactly: `Lab5 final submission`

---
*CMSC395 Computer Networks — Lab 5*  
*Push this notebook to your repository with the message: `Lab5 final submission`*